In [ ]:
%sql
--------------------------------------------------------------------------------------------------------------
--- Epic Clarity Dose Era - Adapted to Databricks SQL from Pure SQL dose_era written by Chris_Knoll:
--- https://gist.github.com/chrisknoll/c820cc12d833db2e3d1e
--- Uses omop_epic schema for Epic Clarity DQD testing
--- INTERVAL set to 30 days
--- Uses concept_relationship (RxNorm has ing) instead of concept_ancestor for ingredient mapping
--- NOTE: Epic drug_exposure does not have dose information; using NULL dose_value
--------------------------------------------------------------------------------------------------------------

TRUNCATE TABLE _exponent.omop_epic.dose_era;

WITH cteDrugTarget AS (
    SELECT
        d.drug_exposure_id,
        d.person_id,
        ing.concept_id AS drug_concept_id,              --- Ingredient-level concept
        0 AS unit_concept_id,                           --- No dose unit in Epic data
        CAST(NULL AS DOUBLE) AS dose_value,             --- No dose value in Epic data
        d.drug_exposure_start_date,
        d.days_supply,

        --- Determine exposure end date
        COALESCE(
            d.drug_exposure_end_date,
            CASE
                WHEN d.days_supply IS NOT NULL AND d.days_supply > 0
                    THEN date_add(d.drug_exposure_start_date, CAST(d.days_supply AS INT))
                ELSE NULL
            END,
            date_add(d.drug_exposure_start_date, 1)
        ) AS drug_exposure_end_date

    FROM _exponent.omop_epic.drug_exposure d
        -- First map NDC → RxNorm via "Maps to"
        LEFT JOIN _exponent.omop.concept_relationship cr_map
          ON cr_map.concept_id_1 = d.drug_concept_id
          AND cr_map.relationship_id = 'Maps to'
        -- Get the RxNorm concept (either mapped or original)
        JOIN _exponent.omop.concept rxnorm
          ON rxnorm.concept_id = COALESCE(cr_map.concept_id_2, d.drug_concept_id)
          AND rxnorm.vocabulary_id = 'RxNorm'
        -- Map RxNorm drug → Ingredient via "RxNorm has ing"
        JOIN _exponent.omop.concept_relationship cr_ing
          ON cr_ing.concept_id_1 = rxnorm.concept_id
          AND cr_ing.relationship_id = 'RxNorm has ing'
        -- Get the ingredient concept
        JOIN _exponent.omop.concept ing
          ON ing.concept_id = cr_ing.concept_id_2
          AND ing.vocabulary_id = 'RxNorm'
          AND ing.concept_class_id = 'Ingredient'
    WHERE d.drug_concept_id != 0
)

-----------------------------------------------------------------------------------------------------------------------------
, cteEndDates AS (
    SELECT
        person_id,
        drug_concept_id,
        unit_concept_id,
        dose_value,
        date_add(event_date, -30) AS end_date
    FROM
    (
        SELECT
            person_id,
            drug_concept_id,
            unit_concept_id,
            dose_value,
            event_date,
            event_type,
            MAX(start_ordinal) OVER (
                PARTITION BY person_id, drug_concept_id, unit_concept_id, dose_value
                ORDER BY event_date, event_type
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS start_ordinal,
            ROW_NUMBER() OVER (
                PARTITION BY person_id, drug_concept_id, unit_concept_id, dose_value
                ORDER BY event_date, event_type
            ) AS overall_ord
        FROM
        (
            SELECT
                person_id,
                drug_concept_id,
                unit_concept_id,
                dose_value,
                drug_exposure_start_date AS event_date,
                -1 AS event_type,
                ROW_NUMBER() OVER (
                    PARTITION BY person_id, drug_concept_id, unit_concept_id, dose_value
                    ORDER BY drug_exposure_start_date
                ) AS start_ordinal
            FROM cteDrugTarget

            UNION ALL

            SELECT
                person_id,
                drug_concept_id,
                unit_concept_id,
                dose_value,
                date_add(drug_exposure_end_date, 30) AS event_date,
                1 AS event_type,
                NULL AS start_ordinal
            FROM cteDrugTarget
        ) RAWDATA
    ) e
    WHERE (2 * e.start_ordinal) - e.overall_ord = 0
)

-----------------------------------------------------------------------------------------------------------------------------
, cteDoseEraEnds AS (
    SELECT
        dt.person_id,
        dt.drug_concept_id,
        dt.unit_concept_id,
        dt.dose_value,
        dt.drug_exposure_start_date,
        MIN(e.end_date) AS dose_era_end_date
    FROM cteDrugTarget dt
    JOIN cteEndDates e
      ON dt.person_id       = e.person_id
     AND dt.drug_concept_id = e.drug_concept_id
     AND dt.unit_concept_id = e.unit_concept_id
     AND dt.dose_value      <=> e.dose_value
     AND e.end_date >= dt.drug_exposure_start_date
    GROUP BY
        dt.person_id,
        dt.drug_concept_id,
        dt.unit_concept_id,
        dt.dose_value,
        dt.drug_exposure_start_date
)

-----------------------------------------------------------------------------------------------------------------------------
--- Final DOSE_ERA insert
-----------------------------------------------------------------------------------------------------------------------------
INSERT INTO _exponent.omop_epic.dose_era (
    person_id,
    drug_concept_id,
    unit_concept_id,
    dose_value,
    dose_era_start_date,
    dose_era_end_date
)
SELECT
    person_id,
    drug_concept_id,
    unit_concept_id,
    dose_value,
    MIN(drug_exposure_start_date) AS dose_era_start_date,
    dose_era_end_date
FROM cteDoseEraEnds
GROUP BY
    person_id,
    drug_concept_id,
    unit_concept_id,
    dose_value,
    dose_era_end_date
ORDER BY
    person_id,
    drug_concept_id
;

In [ ]:
%sql
-- Validation: count dose eras
SELECT COUNT(*) AS dose_era_count FROM _exponent.omop_epic.dose_era

In [ ]:
%sql
-- Validation: top drugs by dose era count
SELECT 
  de.drug_concept_id,
  c.concept_name,
  de.unit_concept_id,
  de.dose_value,
  COUNT(*) AS era_count
FROM _exponent.omop_epic.dose_era de
JOIN _exponent.omop.concept c ON c.concept_id = de.drug_concept_id
GROUP BY de.drug_concept_id, c.concept_name, de.unit_concept_id, de.dose_value
ORDER BY era_count DESC
LIMIT 10